# 1. Dependencies

In [ ]:
import numpy as np
import pennylane as qp
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from model import HybridQuantumCapsuleNetwork

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        print("Found a GPU:", gpu)
else:
    print("No GPU detected by TensorFlow.")

Found a GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


# 2. Dataset

In [3]:
dataset_name = "MNIST"

(x_train, y_train), (x_test , y_test)= tf.keras.datasets.mnist.load_data()

id_tr = np.where(((y_train==0) | (y_train==1)))[0]
id_ts = np.where(((y_test==0) | (y_test==1)))[0]
x_train = x_train[id_tr]
y_train = y_train[id_tr]
x_test = x_test[id_ts]
y_test = y_test[id_ts]

X_train = x_train / 255.0
X_train = tf.cast(X_train, dtype=tf.float32)
X_train = tf.expand_dims(X_train, axis=-1)

X_test = x_test / 255.0
X_test = tf.cast(X_test, dtype=tf.float32)
X_test = tf.expand_dims(X_test, axis=-1)

BatchSize = 16
img_height = 28
img_width = 28
img_dim = 1
img_shape = (img_height, img_width, img_dim)

training_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
training_dataset = training_dataset.batch(batch_size=BatchSize)

validation_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)) # dummy
validation_dataset = validation_dataset.batch(batch_size=BatchSize)

testing_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
testing_dataset = testing_dataset.batch(batch_size=BatchSize)

AUTOTUNE = tf.data.experimental.AUTOTUNE

training_ds = training_dataset.cache().prefetch(buffer_size=AUTOTUNE)
validation_ds = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)
testing_ds = testing_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# 3. Model

In [4]:
num_class = 2
Epohcs = 10
params = {
    "hybrid": True,
    "reconstruction": True,
    "recon_size": (img_shape[0],img_shape[1],1),
    "img_shape": img_shape,
    "num_conv_layer_kernels": 256,
    "num_primary_capsules": 16,
    "num_digitcaps": num_class,
    "num_primary_capsule_dim": 16,
    "num_digitcaps_dim": 8,
    "r": 3,
    "conv_kernel_size": [9,9],
    "conv_strides": [1,1],
    "primary_capsule_kernel_size": [9,9],
    "primary_capsule_strides": [2,2],
    "m_plus" : 0.9, "m_minus" : 0.1, "lambda_" : 0.5, "alpha": 0.0005,
    "n_qubits_pqc": 4, # num_primary_capsule_dim = 2^n_qubits_pqc
    "n_pqc_layer": 3, # n_pqc_layer < n_qubits_pqc
    "pqc_layer": "strongly",
    "imprimitive": qp.ECR,
    "q_embedding": "amplitude", 
    "noise_prop": 0.2, 
    "readout_prob": 0.1
}

model = HybridQuantumCapsuleNetwork(**params)
history = model.fit_custom(training_ds, epochs=Epohcs, val_dataset=validation_ds)

Epoch 1/10: 100%|█| 792/792 [00:46<00:00, 17.06it/s, Loss: 0.4050372 Acc: 0.876273, Val_Loss: 0.405018 Val_Acc: 0.88132
Epoch 2/10: 100%|█| 792/792 [00:34<00:00, 22.65it/s, Loss: 0.4050175 Acc: 0.651954, Val_Loss: 0.405018 Val_Acc: 0.66335
Epoch 3/10: 100%|█| 792/792 [00:34<00:00, 22.82it/s, Loss: 0.40501636 Acc: 0.830715, Val_Loss: 0.405017 Val_Acc: 0.8236
Epoch 4/10: 100%|█| 792/792 [00:34<00:00, 22.72it/s, Loss: 0.40501565 Acc: 0.904619, Val_Loss: 0.405016 Val_Acc: 0.9063
Epoch 5/10: 100%|█| 792/792 [00:35<00:00, 22.59it/s, Loss: 0.40501365 Acc: 0.914805, Val_Loss: 0.405013 Val_Acc: 0.9153
Epoch 6/10: 100%|█| 792/792 [00:34<00:00, 22.97it/s, Loss: 0.40501198 Acc: 0.91741, Val_Loss: 0.405012 Val_Acc: 0.91962
Epoch 7/10: 100%|█| 792/792 [00:34<00:00, 22.87it/s, Loss: 0.4050113 Acc: 0.918358, Val_Loss: 0.405012 Val_Acc: 0.92151
Epoch 8/10: 100%|█| 792/792 [00:34<00:00, 22.78it/s, Loss: 0.40501076 Acc: 0.916779, Val_Loss: 0.405011 Val_Acc: 0.9153
Epoch 9/10: 100%|█| 792/792 [00:34<00:00

# 4. Evaluation

In [5]:
def predictScore(model, x):
    pred = model.safe_norm(model.predict_capsule_output(x))
    pred = tf.squeeze(pred, [1])
    return pred

test_predictions = []
test_labels = []
X_test = []
y_test = []
y_score = []
for data in testing_ds:
    X_test.append(data[0].numpy())
    y_test.append(data[1].numpy())
    preds = model.predict_result(data[0])
    y_score.extend(predictScore(model,data[0]))
    test_predictions.extend(preds)
    test_labels.extend(data[1].numpy())

X_test = tf.concat(X_test,axis=0)
y_test = tf.concat(y_test,axis=0)
test_labels = np.array(test_labels).flatten()
test_predictions = np.array(test_predictions).flatten()
y_score = np.array(y_score)

testResults = {
    "accuracy": accuracy_score(test_labels, test_predictions),
    "precission": precision_score(test_labels, test_predictions),
    "recall": recall_score(test_labels, test_predictions),
    "f1score":f1_score(test_labels, test_predictions),
    "auc": roc_auc_score(test_labels, test_predictions)
}

print("Accuracy\t: ", testResults["accuracy"])
print("Precision\t: ", testResults["precission"])
print("Recall\t\t: ", testResults["recall"])
print("F1-score\t: ", testResults["f1score"])
print("AUC\t\t: ", testResults["auc"])

Accuracy	:  0.9602836879432625
Precision	:  0.9605609114811569
Recall		:  0.9656387665198238
F1-score	:  0.9630931458699473
AUC		:  0.9598601995864425
